# Experiment 2 — YOLO11n Instance Segmentation with Dice Loss

## Objective

This experiment evaluates whether replacing the default BCE-based segmentation loss with **Dice Loss** can improve the segmentation performance of the inventory recognition model.

The original model was trained using:

- Model: YOLO11n-Seg
- Dataset: Original AIIC inventory dataset
- Number of classes: 109
- Image size: 640 × 640
- Epochs: 100
- Batch size: 8
- Patience: 20
- Segmentation loss: Binary Cross-Entropy (BCE)

For this experiment, all major training settings will remain the same to ensure a fair comparison.

The main change is:

**BCE Segmentation Loss → Dice Segmentation Loss**

The final Dice model will later be evaluated using the same test dataset and compared with the original BCE model using:

- Precision
- Recall
- Mask mAP@50
- Mask mAP@50–95
- Validation loss

## 1. Setup and Environment Check

This section prepares the environment for the Dice Loss training experiment.

The purpose of this step is to:

- Import the required libraries such as PyTorch and Ultralytics.
- Define the project, dataset, and training output paths.
- Check the installed Ultralytics and PyTorch versions.
- Verify that CUDA is available for GPU training.
- Confirm that the correct dataset configuration file (`data.yaml`) can be found.

No model training is performed in this step. This is only an environment and path verification before modifying the YOLO segmentation loss function.

In [1]:
# ============================================================
# CELL 1 — SETUP & ENVIRONMENT CHECK
# ============================================================

from pathlib import Path
import torch
import ultralytics
from ultralytics import YOLO

project_path = Path(r"G:\AIIC")

segmentation_project = (
    project_path
    / "yolo_segmentation"
)

data_yaml = (
    segmentation_project
    / "dataset"
    / "data.yaml"
)

runs_path = (
    segmentation_project
    / "runs"
)

print("=" * 70)
print("DICE LOSS EXPERIMENT — ENVIRONMENT CHECK")
print("=" * 70)

print(f"Ultralytics version : {ultralytics.__version__}")
print(f"PyTorch version     : {torch.__version__}")
print(f"CUDA available      : {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"GPU                 : {torch.cuda.get_device_name(0)}")

print()
print(f"Project path        : {project_path}")
print(f"Dataset YAML        : {data_yaml}")
print(f"Dataset YAML exists : {data_yaml.exists()}")
print(f"Runs folder         : {runs_path}")

print("=" * 70)

DICE LOSS EXPERIMENT — ENVIRONMENT CHECK
Ultralytics version : 8.4.137
PyTorch version     : 2.13.0+cu130
CUDA available      : True
GPU                 : NVIDIA GeForce RTX 5060

Project path        : G:\AIIC
Dataset YAML        : G:\AIIC\yolo_segmentation\dataset\data.yaml
Dataset YAML exists : True
Runs folder         : G:\AIIC\yolo_segmentation\runs


## 2. Inspect the Current YOLO Segmentation Loss

Before replacing the segmentation loss with Dice Loss, the existing YOLO11 segmentation loss implementation must first be inspected.

This step is important because different Ultralytics versions may implement the segmentation loss differently.

The purpose of this section is to:

- Locate the current YOLO instance segmentation loss function.
- Confirm that the current mask loss uses Binary Cross-Entropy (BCE).
- Identify the exact function signature that must be preserved when replacing BCE with Dice Loss.

No training is performed in this step.

In [2]:
# ============================================================
# CELL 2 — INSPECT CURRENT YOLO SEGMENTATION LOSS
# ============================================================

import inspect

from ultralytics.utils.loss import v8SegmentationLoss

print("=" * 70)
print("CURRENT YOLO INSTANCE SEGMENTATION LOSS FUNCTION")
print("=" * 70)

# Get the source code of the current mask loss function
mask_loss_source = inspect.getsource(
    v8SegmentationLoss.single_mask_loss
)

print(mask_loss_source)

print("=" * 70)

# Simple automatic check
if "binary_cross_entropy" in mask_loss_source.lower():
    print("✅ BCE-based instance mask loss detected.")
else:
    print("⚠️ BCE was not detected automatically.")
    print("Please inspect the function above manually.")

CURRENT YOLO INSTANCE SEGMENTATION LOSS FUNCTION
    @staticmethod
    def single_mask_loss(
        gt_mask: torch.Tensor, pred: torch.Tensor, proto: torch.Tensor, xyxy: torch.Tensor, area: torch.Tensor
    ) -> torch.Tensor:
        """Compute the instance segmentation loss for a single image.

        Args:
            gt_mask (torch.Tensor): Ground truth mask of shape (N, H, W), where N is the number of objects.
            pred (torch.Tensor): Predicted mask coefficients of shape (N, 32).
            proto (torch.Tensor): Prototype masks of shape (32, H, W).
            xyxy (torch.Tensor): Ground truth bounding boxes in xyxy format, normalized to [0, 1], of shape (N, 4).
            area (torch.Tensor): Area of each ground truth bounding box of shape (N,).

        Returns:
            (torch.Tensor): The calculated mask loss for a single image.

        Notes:
            The function uses the equation pred_mask = torch.einsum('in,nhw->ihw', pred, proto) to produce the
         

## 3. Replace BCE Mask Loss with Dice Loss

The default YOLO11 instance segmentation model uses **Binary Cross-Entropy (BCE) with logits** for the instance mask loss.

In this experiment, only the segmentation mask loss is replaced with **Dice Loss**.

The other YOLO loss components remain unchanged:

- Bounding box loss — unchanged
- Classification loss — unchanged
- Distribution Focal Loss (DFL) — unchanged
- Segmentation mask loss — changed from BCE to Dice Loss

Dice Loss measures the overlap between the predicted mask and the ground-truth mask.

The Dice coefficient is defined as:

\[
Dice = \frac{2|P \cap G| + \epsilon}{|P| + |G| + \epsilon}
\]

where:

- \(P\) = predicted segmentation mask
- \(G\) = ground-truth segmentation mask
- \(\epsilon\) = small constant for numerical stability

The Dice Loss is:

\[
L_{Dice} = 1 - Dice
\]

A lower Dice Loss indicates greater overlap between the predicted and ground-truth masks.

This experiment uses **pure Dice Loss**, not a combination of BCE and Dice Loss.

### Implementation Note

The initial Dice Loss implementation applied `crop_mask()` directly to the predicted mask probabilities produced by the sigmoid function.

Ultralytics' `crop_mask()` performs an in-place operation. This modified a tensor required by PyTorch for gradient computation and caused an autograd error during backpropagation.

To solve this, the bounding-box crop region is generated separately using a tensor of ones. The predicted and ground-truth masks are then multiplied by this crop region without modifying the sigmoid output in-place.

This preserves the computation graph required for backpropagation.

In [6]:
# ============================================================
# CELL 3 — REPLACE BCE MASK LOSS WITH PURE DICE LOSS
# AUTOGRAD-SAFE VERSION
# ============================================================

import torch

from ultralytics.utils.loss import v8SegmentationLoss
from ultralytics.utils.ops import crop_mask


# ------------------------------------------------------------
# SAVE ORIGINAL BCE LOSS
# ------------------------------------------------------------

if not hasattr(v8SegmentationLoss, "_original_single_mask_loss"):
    v8SegmentationLoss._original_single_mask_loss = (
        v8SegmentationLoss.single_mask_loss
    )


# ------------------------------------------------------------
# PURE DICE MASK LOSS
# ------------------------------------------------------------

def dice_single_mask_loss(
    gt_mask: torch.Tensor,
    pred: torch.Tensor,
    proto: torch.Tensor,
    xyxy: torch.Tensor,
    area: torch.Tensor
) -> torch.Tensor:

    """
    Compute Pure Dice Loss for YOLO11 instance segmentation.

    This function replaces only the instance mask loss.
    Bounding box, classification and DFL losses remain unchanged.
    """

    # --------------------------------------------------------
    # 1. BUILD PREDICTED MASK LOGITS
    # --------------------------------------------------------

    pred_mask = torch.einsum(
        "in,nhw->ihw",
        pred,
        proto
    )

    # Convert logits to probabilities
    pred_prob = pred_mask.float().sigmoid()

    # Ensure GT mask is float
    gt_mask = gt_mask.float()


    # --------------------------------------------------------
    # 2. CREATE BOUNDING-BOX CROP REGION
    # --------------------------------------------------------
    #
    # We do NOT apply crop_mask() directly to pred_prob.
    # crop_mask() uses in-place modification, which can break
    # PyTorch autograd when applied to sigmoid output.
    #
    # Instead, create a separate crop region using ones.
    # --------------------------------------------------------

    crop_region = torch.ones_like(pred_prob)

    crop_region = crop_mask(
        crop_region,
        xyxy
    )


    # --------------------------------------------------------
    # 3. APPLY CROP WITHOUT IN-PLACE MODIFICATION
    # --------------------------------------------------------

    pred_crop = pred_prob * crop_region
    gt_crop = gt_mask * crop_region


    # --------------------------------------------------------
    # 4. CALCULATE DICE COEFFICIENT
    # --------------------------------------------------------

    intersection = (
        pred_crop * gt_crop
    ).sum(dim=(1, 2))

    pred_sum = pred_crop.sum(dim=(1, 2))
    gt_sum = gt_crop.sum(dim=(1, 2))

    eps = 1e-7

    dice_score = (
        2.0 * intersection + eps
    ) / (
        pred_sum + gt_sum + eps
    )


    # --------------------------------------------------------
    # 5. CALCULATE DICE LOSS
    # --------------------------------------------------------

    dice_loss = 1.0 - dice_score

    # Sum loss across all instances in the image
    return dice_loss.sum()


# ------------------------------------------------------------
# APPLY CUSTOM DICE LOSS
# ------------------------------------------------------------

v8SegmentationLoss.single_mask_loss = staticmethod(
    dice_single_mask_loss
)


# ------------------------------------------------------------
# CONFIRMATION
# ------------------------------------------------------------

print("=" * 70)
print("CUSTOM SEGMENTATION LOSS ACTIVATED")
print("=" * 70)

print("Original mask loss : BCEWithLogits")
print("New mask loss      : Pure Dice Loss")

print()
print("Box loss           : unchanged")
print("Classification loss: unchanged")
print("DFL loss           : unchanged")

print("=" * 70)
print("✅ AUTOGRAD-SAFE PURE DICE LOSS ACTIVATED")

CUSTOM SEGMENTATION LOSS ACTIVATED
Original mask loss : BCEWithLogits
New mask loss      : Pure Dice Loss

Box loss           : unchanged
Classification loss: unchanged
DFL loss           : unchanged
✅ AUTOGRAD-SAFE PURE DICE LOSS ACTIVATED


## 4. Verify the Custom Dice Loss

Before starting the training process, the active segmentation loss function is verified.

This step ensures that:

- The YOLO instance segmentation model is using the custom Dice Loss function.
- The original BCE mask loss is no longer active for the segmentation component.
- The modification only affects the instance mask loss.

This verification is important because the training experiment must genuinely use Dice Loss before it can be compared with the original BCE baseline.

In [8]:
# ============================================================
# CELL 4 — VERIFY ACTIVE SEGMENTATION LOSS
# ============================================================

from ultralytics.utils.loss import v8SegmentationLoss

active_loss = v8SegmentationLoss.single_mask_loss

print("=" * 70)
print("ACTIVE SEGMENTATION LOSS VERIFICATION")
print("=" * 70)

print(f"Active function name : {active_loss.__name__}")

# Check functions referenced inside the active loss
function_names = active_loss.__code__.co_names

uses_bce = "binary_cross_entropy_with_logits" in function_names
uses_dice_function = active_loss.__name__ == "dice_single_mask_loss"

print(f"BCE function detected : {uses_bce}")
print(f"Dice function active  : {uses_dice_function}")

print("=" * 70)

if uses_dice_function and not uses_bce:
    print("✅ VERIFIED: PURE DICE LOSS IS ACTIVE")
    print("✅ BCE mask loss is NOT being used")
else:
    print("❌ VERIFICATION FAILED")
    print("Do NOT start training yet.")

ACTIVE SEGMENTATION LOSS VERIFICATION
Active function name : dice_single_mask_loss
BCE function detected : False
Dice function active  : True
✅ VERIFIED: PURE DICE LOSS IS ACTIVE
✅ BCE mask loss is NOT being used


## 5. Dice Loss Smoke Test

Before performing the full 100-epoch training, a one-epoch smoke test is conducted.

The purpose of this test is to confirm that:

- The custom Dice Loss is compatible with the YOLO11 segmentation training pipeline.
- Forward propagation and backpropagation can be completed successfully.
- GPU training operates normally.
- No tensor shape, mask, or loss calculation errors occur.

The smoke-test model is only used for technical verification and will not be used as the final Dice model.

After the smoke test succeeds, the full experiment will restart from the original pretrained `yolo11n-seg.pt` weights.

In [9]:
# ============================================================
# CELL 5 — DICE LOSS SMOKE TEST
# 1 EPOCH ONLY
# ============================================================

from ultralytics import YOLO

print("=" * 70)
print("STARTING DICE LOSS SMOKE TEST")
print("=" * 70)

# Fresh pretrained model
smoke_model = YOLO("yolo11n-seg.pt")

smoke_results = smoke_model.train(
    data=str(data_yaml),

    epochs=1,
    imgsz=640,
    batch=8,

    device=0,
    workers=4,

    patience=20,

    pretrained=True,
    optimizer="auto",

    amp=True,
    cache=False,

    project=str(runs_path),
    name="dice_smoke_test",

    exist_ok=True,

    plots=True,
    verbose=True
)

print()
print("=" * 70)
print("✅ DICE LOSS SMOKE TEST COMPLETED")
print("=" * 70)

STARTING DICE LOSS SMOKE TEST
New https://pypi.org/project/ultralytics/8.4.143 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.137  Python-3.14.3 torch-2.13.0+cu130 CUDA:0 (NVIDIA GeForce RTX 5060, 8123MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=G:\AIIC\yolo_segmentation\dataset\data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=1, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.

## 6. Full Training with Dice Loss

After successfully completing the one-epoch smoke test, the custom Dice Loss implementation is confirmed to be compatible with the YOLO11 instance segmentation training pipeline.

The full Dice Loss experiment is now performed using the same major training configuration as the original BCE baseline to ensure a fair comparison.

### Training Configuration

- Model: YOLO11n-Seg
- Initial weights: `yolo11n-seg.pt`
- Dataset: Original dataset
- Number of classes: 109
- Image size: 640 × 640
- Maximum epochs: 100
- Batch size: 8
- Early stopping patience: 20
- Optimizer: Auto
- GPU: CUDA device 0
- Segmentation mask loss: Pure Dice Loss

The model is initialized again from the original pretrained YOLO11n-Seg weights rather than continuing from the smoke-test model.

This ensures that the Dice experiment is trained independently and can be fairly compared with the original BCE baseline.

In [10]:
# ============================================================
# CELL 6 — FULL DICE LOSS TRAINING
# MODEL B: ORIGINAL DATASET + PURE DICE LOSS
# ============================================================

from ultralytics import YOLO

print("=" * 70)
print("MODEL B — FULL DICE LOSS TRAINING")
print("=" * 70)

print("Model        : YOLO11n-Seg")
print("Dataset      : Original AIIC Dataset")
print("Mask Loss    : Pure Dice Loss")
print("Epochs       : 100")
print("Batch Size   : 8")
print("Image Size   : 640")
print("Patience     : 20")

print("=" * 70)


# ------------------------------------------------------------
# LOAD FRESH PRETRAINED MODEL
# ------------------------------------------------------------
#
# IMPORTANT:
# Start again from the original YOLO pretrained weights.
#
# Do NOT use:
# - BCE best.pt
# - Dice smoke-test weights
# ------------------------------------------------------------

model_dice = YOLO("yolo11n-seg.pt")


# ------------------------------------------------------------
# START FULL TRAINING
# ------------------------------------------------------------

dice_results = model_dice.train(

    data=str(data_yaml),

    epochs=100,
    imgsz=640,
    batch=8,

    device=0,
    workers=4,

    patience=20,

    pretrained=True,
    optimizer="auto",

    amp=True,
    cache=False,

    project=str(runs_path),

    # Separate folder from BCE baseline
    name="yolo11n_seg_aiic_dice_original",

    # Prevent accidental overwrite of an existing experiment
    exist_ok=False,

    plots=True,
    verbose=True
)


print()
print("=" * 70)
print("✅ FULL DICE LOSS TRAINING COMPLETED")
print("=" * 70)

MODEL B — FULL DICE LOSS TRAINING
Model        : YOLO11n-Seg
Dataset      : Original AIIC Dataset
Mask Loss    : Pure Dice Loss
Epochs       : 100
Batch Size   : 8
Image Size   : 640
Patience     : 20
New https://pypi.org/project/ultralytics/8.4.144 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.137  Python-3.14.3 torch-2.13.0+cu130 CUDA:0 (NVIDIA GeForce RTX 5060, 8123MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=G:\AIIC\yolo_segmentation\dataset\data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0